# 🔬 Experimento M2 · ¿el colapso de HyperNCD es un problema de dominio?

**Pregunta:** en la capilla, M2 manda ~98 % de los puntos a una sola cabeza y su `f_sem` casi no tiene variedad (rango efectivo 3,4). ¿Es porque el modelo se entrenó en otro dominio (SemanticPOSS), o porque algo falla en el checkpoint o en cómo extraemos `f_sem`?

**Cómo se responde:** el MISMO checkpoint y el MISMO código de inferencia (`predecir_hyperncd.py`) sobre:
1. **SemanticPOSS** (secuencia 03 = validación, y 00 = entrenamiento): el dominio donde se entrenó. *Control positivo.*
2. **La capilla etiquetada** y la **capilla con la densidad de un LiDAR** (80 000 puntos al azar): separa "dominio" de "densidad de puntos".
3. **PL7 y PL13** (Catedral), con los `f_sem` ya guardados.
4. Tres **checkpoints** de la misma corrida (`best` época 8, `best` época 4 y `last`): descarta que el elegido sea el malo.

**Cómo leer el resultado:**
- Si en **SemanticPOSS también colapsa** (cabeza dominante muy alta, rango bajo, NMI cerca de 0) → el problema NO es el dominio: es el entrenamiento, el checkpoint o la extracción.
- Si en **SemanticPOSS se reparte bien** y en la capilla colapsa → apoya "brecha de dominio". Si además la capilla submuestreada mejora, parte de la brecha es la densidad.

**Requiere:** GPU (T4 sirve) y que el cuaderno 1 ya haya guardado el entorno `hyperncd_env_cache.tar.gz` en tu Drive. Dura unos minutos. No modifica nada tuyo: guarda en `m2/salida/experimento_dominio/`.

In [ ]:
# @title 1. GPU, Drive y rutas
!nvidia-smi --query-gpu=name,memory.total --format=csv
import os, glob
from google.colab import drive
drive.mount('/content/drive')
DRIVE = "/content/drive/MyDrive/PFC1_HyperNCD_Sonata"
M1_SAL, M2_ENT, M2_SAL = f"{DRIVE}/m1/salida", f"{DRIVE}/m2/entrada", f"{DRIVE}/m2/salida"
ESCENA = "capilla_etiquetada"
N_ESCANEOS = 8            # escaneos de SemanticPOSS por secuencia
SUBMUESTREO = 80000       # puntos de la capilla "a densidad de LiDAR"
BRANCH = "main"
SAL = f"{M2_SAL}/experimento_dominio"; os.makedirs(SAL, exist_ok=True)
WORK = "/content/work"; os.makedirs(WORK, exist_ok=True)
print("Resultados se guardan en:", SAL)

In [ ]:
# @title 2. Clonar el repo y asegurar el script del experimento
import os
REPO_OWNER, REPO_NAME = "manyorycuevamendoza", "PFC1-HyperNCD-Sonata"
REPO_DIR = f"/content/{REPO_NAME}"
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
except Exception:
    token = None
if not token:
    from getpass import getpass
    token = getpass("GitHub token (oculto): ").strip()
if not os.path.exists(REPO_DIR):
    !git clone -q -b {BRANCH} https://{token}@github.com/{REPO_OWNER}/{REPO_NAME}.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q origin {BRANCH}
SCRIPT = f"{REPO_DIR}/experimentacion/m2_dominio/medir_m2.py"
if not os.path.exists(SCRIPT):
    print("El script aún no está en GitHub. Súbelo ahora desde tu laptop: experimentacion/m2_dominio/medir_m2.py")
    from google.colab import files
    import shutil
    sub = files.upload()
    os.makedirs(os.path.dirname(SCRIPT), exist_ok=True)
    shutil.move(next(iter(sub)), SCRIPT)
assert os.path.exists(f"{REPO_DIR}/predecir_hyperncd.py"), "Falta predecir_hyperncd.py en el repo"
print("✅ listo:", SCRIPT)

In [ ]:
# @title 3. Restaurar hyperncd-env + MinkowskiEngine (de tu Drive, ~20 s)
%%bash
set -e

ENV_CACHE="/content/drive/MyDrive/PFC1_HyperNCD_Sonata/entornos/hyperncd_env_cache.tar.gz"

if [ ! -d "/content/miniconda3" ]; then
    echo "[1/4] Instalando Miniconda3 base..."
    wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O /tmp/miniconda.sh
    bash /tmp/miniconda.sh -b -p /content/miniconda3
fi

source /content/miniconda3/etc/profile.d/conda.sh
conda config --set always_yes yes
conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main 2>/dev/null || true
conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r 2>/dev/null || true

# Si ya existe la imagen del entorno compilada en tu Google Drive, la restauramos en ~20 segundos
if [ -f "$ENV_CACHE" ] && [ ! -d "/content/miniconda3/envs/hyperncd-env" ]; then
    echo "⚡ Encontrado entorno pre-compilado en tu Drive ($ENV_CACHE). Restaurando en ~20 segundos..."
    mkdir -p /content/miniconda3/envs/hyperncd-env
    tar -xzf "$ENV_CACHE" -C /content/miniconda3/envs/hyperncd-env
    /content/miniconda3/envs/hyperncd-env/bin/python -c "import torch, MinkowskiEngine as ME; print('✅ Entorno restaurado desde Drive en segundos! MinkowskiEngine:', ME.__version__)"
    exit 0
fi

if [ ! -d "/content/miniconda3/envs/hyperncd-env" ]; then
    echo "[2/4] Creando entorno hyperncd-env (Python 3.8 + CUDA 11.8 + OpenBLAS)..."
    conda create -y -n hyperncd-env python=3.8 -c conda-forge
    conda install -y -n hyperncd-env -c nvidia/label/cuda-11.8.0 cuda-nvcc cuda-cudart-dev cuda-libraries-dev
    conda install -y -n hyperncd-env -c conda-forge openblas-devel ninja gxx_linux-64=11
fi

conda activate hyperncd-env

echo "[3/4] Instalando PyTorch 1.10.1+cu113 y dependencias de HyperNCD..."
pip install -q torch==1.10.1+cu113 torchvision==0.11.2+cu113 -f https://download.pytorch.org/whl/torch_stable.html
pip install -q pytorch-lightning==1.4.8 torchmetrics==0.7.2 scipy==1.7.3 pyyaml tqdm scikit-learn plotly matplotlib
pip install -q torch-scatter -f https://data.pyg.org/whl/torch-1.10.0+cu113.html

if ! python -c "import MinkowskiEngine" 2>/dev/null; then
    echo "[4/4] Compilando e instalando MinkowskiEngine (solo esta primera vez)..."
    # Asegurar cabecera nvtx3/nvToolsExt.h que pide MinkowskiEngine
    conda install -y -n hyperncd-env -c nvidia/label/cuda-11.8.0 cuda-nvtx-dev 2>/dev/null || true
    if [ ! -f "$CONDA_PREFIX/include/nvtx3/nvToolsExt.h" ]; then
        git clone -q --depth 1 https://github.com/NVIDIA/NVTX.git /tmp/NVTX
        cp -r /tmp/NVTX/c/include/nvtx3 "$CONDA_PREFIX/include/"
        rm -rf /tmp/NVTX
    fi
    if [ ! -d "/content/MinkowskiEngine" ]; then
        git clone -q https://github.com/NVIDIA/MinkowskiEngine.git /content/MinkowskiEngine
    fi
    cd /content/MinkowskiEngine
    export CUDA_HOME="$CONDA_PREFIX"
    export PATH="$CONDA_PREFIX/bin:$PATH"
    export CPLUS_INCLUDE_PATH="$CONDA_PREFIX/include:${CPLUS_INCLUDE_PATH:-}"
    export TORCH_CUDA_ARCH_LIST="7.5;8.0;8.6"
    export MAX_JOBS=4
    export CC="$CONDA_PREFIX/bin/x86_64-conda-linux-gnu-gcc"
    export CXX="$CONDA_PREFIX/bin/x86_64-conda-linux-gnu-g++"
    python setup.py install --blas_include_dirs=${CONDA_PREFIX}/include --blas=openblas
fi

python -c "import torch, MinkowskiEngine as ME; print('✅ MinkowskiEngine listo, versión:', ME.__version__, '| GPU:', torch.cuda.get_device_name(0))"

# Guardar imagen del entorno ya compilado en tu Google Drive para no volver a compilar nunca más
if [ -d "/content/drive/MyDrive" ] && [ ! -f "$ENV_CACHE" ]; then
    echo "💾 Guardando imagen del entorno compilado en tu Drive ($ENV_CACHE) para que la próxima vez cargue en 20 segundos..."
    mkdir -p "$(dirname "$ENV_CACHE")"
    tar -czf "$ENV_CACHE" -C /content/miniconda3/envs/hyperncd-env .
    echo "✅ Imagen del entorno guardada en Drive."
fi

In [ ]:
# @title 4. Correr el experimento con cada checkpoint
import glob, os, subprocess, json
PY_M2 = "/content/miniconda3/envs/hyperncd-env/bin/python"

# --- checkpoints de la corrida split0_run4 (el correcto es el primero) ---
ck = glob.glob(f"{M2_ENT}/**/split0_run4/*.ckpt", recursive=True)
def uno(nombre):
    r = [c for c in ck if os.path.basename(c) == nombre]
    return r[0] if r else None
CKPTS = {"epoca8_best": uno("SemanticPOSS_S0_best-epoch=8-step=11196.ckpt"),
         "epoca4_best": uno("SemanticPOSS_S0_best-epoch=4-step=6220.ckpt"),
         "last": uno("last.ckpt")}
assert CKPTS["epoca8_best"], "No encuentro el checkpoint de la época 8 en split0_run4. Hay:\n" + "\n".join(sorted(ck))
for k, v in CKPTS.items(): print(f"{k:12s}", v or "(no está, se omite)")

# --- datos ---
seq = glob.glob(f"{M2_ENT}/**/SemanticPOSS/dataset/sequences", recursive=True)
assert seq, "No encuentro SemanticPOSS/dataset/sequences dentro de m2/entrada (¿se movió la carpeta datos_m2_descomprimidos?)"
SEQ = seq[0]; print("SemanticPOSS:", SEQ, sorted(os.listdir(SEQ)))
M1 = f"{M1_SAL}/{ESCENA}"
capilla = f"{M1}/{ESCENA}.npy,{M1}/{ESCENA}_etiquetas.npy,{M1}/{ESCENA}_etiquetas.json"
for f in (f"{M1}/{ESCENA}.npy", f"{M1}/{ESCENA}_etiquetas.npy", f"{M1}/{ESCENA}_etiquetas.json"):
    assert os.path.exists(f), f"Falta {f}: corre el cuaderno 1"
pl = {n: glob.glob(f"{M2_SAL}/**/{n}_fsem.npz", recursive=True) for n in ("PL13", "PL7")}
print("f_sem guardados de la Catedral:", {k: (v[0] if v else None) for k, v in pl.items()})

for etiqueta, ruta in CKPTS.items():
    if not ruta: continue
    print("\n" + "=" * 70 + f"\n  CHECKPOINT {etiqueta}\n" + "=" * 70, flush=True)
    cmd = [PY_M2, SCRIPT, "--ckpt", ruta, "--salida", f"{SAL}/{etiqueta}", "--etiqueta", etiqueta, "--n-scans", str(N_ESCANEOS),
           "--semposs", f"semposs_val_03={SEQ}/03", "--semposs", f"semposs_train_00={SEQ}/00",
           "--npy", f"capilla={capilla}", "--submuestreo", str(SUBMUESTREO)]
    if etiqueta == "epoca8_best":                         # los f_sem guardados de PL7/PL13 son de esta época
        for n, v in pl.items():
            if v: cmd += ["--fsem", f"{n}={v[0]}"]
    r = subprocess.run(cmd, cwd=REPO_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    print(r.stdout[-6000:]); assert r.returncode == 0, "falló: mira el error de arriba"
print("\n✅ terminado. Siguiente celda: gráficos.")

In [ ]:
# @title 5. Ver los resultados (gráficos y tabla comparativa)
import json, os
from IPython.display import Image, display, Markdown
etiquetas = [e for e in ("epoca8_best", "epoca4_best", "last") if os.path.exists(f"{SAL}/{e}/resultados.json")]

# tabla: escena x checkpoint
datos = {e: {s["nombre"]: s for s in json.load(open(f"{SAL}/{e}/resultados.json"))["escenas"]} for e in etiquetas}
escenas = list(datos[etiquetas[0]].keys())
f = lambda x, d=1: "-" if x is None else f"{x:.{d}f}"
cab = "| Escena | " + " | ".join(f"{e}: dominante % / rango / NMI" for e in etiquetas) + " |\n|---|" + "---|" * len(etiquetas)
filas = []
for s in escenas:
    celdas = []
    for e in etiquetas:
        r = datos[e].get(s)
        celdas.append("-" if not r else f"{f(r['pct_dominante'])} / {f(r['rango_efectivo'])} / {f((r.get('todas') or {}).get('NMI'), 3)}")
    filas.append(f"| {s} | " + " | ".join(celdas) + " |")
display(Markdown("## Comparación\n\n" + cab + "\n" + "\n".join(filas) +
    "\n\n*Dominante % alto (≈100) y rango bajo (≈1) = colapso. NMI ≈ 0 = las cabezas no separan las clases reales.*"))

for e in etiquetas:
    display(Markdown(f"## Checkpoint `{e}`"))
    for g in ("m2_dominio.png", "m2_cabezas.png", "m2_cabeza_x_clase.png"):
        if os.path.exists(f"{SAL}/{e}/{g}"): display(Image(f"{SAL}/{e}/{g}"))

In [ ]:
# @title 6. Empaquetar lo pequeño y bajarlo a tu laptop (gráficos + números, pocos MB)
import zipfile, os
from google.colab import files
zp = "/content/experimento_m2_dominio.zip"
with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
    for raiz, _, fs in os.walk(SAL):
        for n in fs:
            if n.endswith((".png", ".json", ".md")):
                z.write(os.path.join(raiz, n), os.path.relpath(os.path.join(raiz, n), SAL))
print(round(os.path.getsize(zp) / 2**20, 2), "MB")
files.download(zp)